# Uygulamalı Yapay Zeka - Dönem Projesi (Faz 1)
**Öğrenci:** Edanur Lazoğlu - 241230002

Bu çalışmada UCI Machine Learning Repository üzerinden **Heart Disease (Cleveland - ID: 45)** ve **Statlog (Heart - ID: 145)** veri setleri incelenmekte, keşifsel veri analizi (EDA) yapılmakta ve Faz 2'deki model eğitimi için ön işleme (preprocessing) boru hattı kurulmaktadır.

In [21]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Grafikleri ve çıktı dosyalarını bu klasöre koyacağım
KLASOR = "ciktilar"
os.makedirs(KLASOR, exist_ok=True)

# İki veri setinde de aynı 13 özellik aynı sırayla geliyor.
SUTUNLAR = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

# Sayısal olanlar (ölçekleme yapacağım) ve kategorik olanlar (one-hot yapacağım)
SAYISAL = ["age", "trestbps", "chol", "thalach", "oldpeak"]
KATEGORIK = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]


## 1. Veri Setlerinin İndirilmesi ve İkili (Binary) Hedef Dönüşümü

In [22]:
import ssl
import urllib.request

# 1. Doğrulamasız SSL bağlamı oluştur ve hafızadaki eski bağlantıyı bununla ez
ssl_context = ssl._create_unverified_context()
ssl._create_default_https_context = ssl._create_unverified_context
urllib.request.install_opener(
    urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_context))
)

def veriyi_yukle(uci_id):
    veri = fetch_ucirepo(id=uci_id)             # UCI sitesinden indiriyor
    X = veri.data.features.copy()               # özellikler
    y_ham = veri.data.targets.iloc[:, 0].copy() # hedef değişken (orijinal hali)

    assert X.shape[1] == 13, "13 özellik bekliyordum ama farklı geldi"
    X.columns = SUTUNLAR

    # Heart Disease'te hedef 0-4 arası: 0 hasta değil, 1-4 hasta
    # Statlog'ta hedef 1-2 arası: 1 hasta değil, 2 hasta
    if uci_id == 45:
        y = (y_ham > 0).astype(int)
    else:
        y = (y_ham == 2).astype(int)
    return X, y_ham, y

X1, y1_ham, y1 = veriyi_yukle(45)    # Heart Disease (Cleveland)
X2, y2_ham, y2 = veriyi_yukle(145)   # Statlog (Heart)
print(f"Heart Disease boyutu: {X1.shape} | Statlog Heart boyutu: {X2.shape}")

Heart Disease boyutu: (303, 13) | Statlog Heart boyutu: (270, 13)


## 2. Keşifsel Veri Analizi (EDA) ve Görselleştirmeler

In [23]:
def incele(X, y_ham, y, ad):
    print("\n" + "=" * 60)
    print(ad, "VERİ SETİ ANALİZİ")
    print("=" * 60)

    print("Satır, sütun sayısı:", X.shape)
    print("\nVeri tipleri:")
    print(X.dtypes)

    eksik = X.isnull().sum()
    print("\nEksik değer sayısı (sütun bazında):")
    print(eksik[eksik > 0] if eksik.sum() > 0 else "Hiç eksik değer yok")

    print("\nTekrar eden satır sayısı:", X.duplicated().sum())

    print("\nOrijinal hedef dağılımı:")
    print(y_ham.value_counts().sort_index())
    print("\nİkili hedef (0 = hasta değil, 1 = hasta):")
    print(y.value_counts().sort_index())
    print("Oranlar (%):")
    print((y.value_counts(normalize=True).sort_index() * 100).round(1))

    print("\nSayısal değişkenlerin istatistikleri:")
    ozet = X[SAYISAL].describe().T
    print(ozet.round(2))
    ozet.to_csv(f"{KLASOR}/{ad}_sayisal_ozet.csv")

    print("\nAykırı değer sayıları (IQR yöntemi):")
    for s in SAYISAL:
        q1 = X[s].quantile(0.25)
        q3 = X[s].quantile(0.75)
        iqr = q3 - q1
        alt = q1 - 1.5 * iqr
        ust = q3 + 1.5 * iqr
        sayi = ((X[s] < alt) | (X[s] > ust)).sum()
        print(f"  {s}: {sayi}")

    # Grafik 1: sınıf dağılımı
    plt.figure(figsize=(5, 4))
    sns.countplot(x=y)
    plt.title(f"{ad} - Sınıf dağılımı")
    plt.xlabel("0 = hasta değil, 1 = hasta")
    plt.ylabel("Hasta sayısı")
    plt.tight_layout()
    plt.savefig(f"{KLASOR}/{ad}_1_sinif_dagilimi.png", dpi=150)
    plt.show()

    # Grafik 2: sayısal değişkenlerin histogramları
    fig, eksenler = plt.subplots(2, 3, figsize=(13, 7))
    eksenler = eksenler.flatten()
    for i, s in enumerate(SAYISAL):
        sns.histplot(X[s], kde=True, ax=eksenler[i])
        eksenler[i].set_title(s)
    eksenler[5].axis("off")
    plt.suptitle(f"{ad} - Sayısal değişkenlerin dağılımı")
    plt.tight_layout()
    plt.savefig(f"{KLASOR}/{ad}_2_histogramlar.png", dpi=150)
    plt.show()

    # Grafik 3: sınıfa göre boxplot
    fig, eksenler = plt.subplots(2, 3, figsize=(13, 7))
    eksenler = eksenler.flatten()
    for i, s in enumerate(SAYISAL):
        sns.boxplot(x=y, y=X[s], ax=eksenler[i])
        eksenler[i].set_title(s)
        eksenler[i].set_xlabel("Hastalık (0/1)")
    eksenler[5].axis("off")
    plt.suptitle(f"{ad} - Sınıfa göre boxplot")
    plt.tight_layout()
    plt.savefig(f"{KLASOR}/{ad}_3_boxplot.png", dpi=150)
    plt.show()

    # Grafik 4: kategorik değişkenler
    fig, eksenler = plt.subplots(2, 4, figsize=(17, 7))
    eksenler = eksenler.flatten()
    for i, s in enumerate(KATEGORIK):
        sns.countplot(x=X[s], hue=y, ax=eksenler[i])
        eksenler[i].set_title(s)
        eksenler[i].legend(title="Hastalık", fontsize=7)
    plt.suptitle(f"{ad} - Kategorik değişkenler")
    plt.tight_layout()
    plt.savefig(f"{KLASOR}/{ad}_4_kategorikler.png", dpi=150)
    plt.show()

    # Grafik 5: korelasyon ısı haritası
    tablo = X.copy()
    tablo["hedef"] = y
    plt.figure(figsize=(10, 8))
    sns.heatmap(tablo.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title(f"{ad} - Korelasyon matrisi")
    plt.tight_layout()
    plt.savefig(f"{KLASOR}/{ad}_5_korelasyon.png", dpi=150)
    plt.show()

    # Grafik 6: eksik değerler
    plt.figure(figsize=(8, 4))
    X.isnull().sum().plot(kind="bar")
    plt.title(f"{ad} - Sütunlara göre eksik değer sayısı")
    plt.ylabel("Eksik sayısı")
    plt.tight_layout()
    plt.savefig(f"{KLASOR}/{ad}_6_eksik_degerler.png", dpi=150)
    plt.show()

    print("\nGrafikler kaydedildi ->", KLASOR)

In [24]:
incele(X1, y1_ham, y1, "HeartDisease")
incele(X2, y2_ham, y2, "StatlogHeart")


HeartDisease VERİ SETİ ANALİZİ
Satır, sütun sayısı: (303, 13)

Veri tipleri:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
dtype: object

Eksik değer sayısı (sütun bazında):
ca      4
thal    2
dtype: int64

Tekrar eden satır sayısı: 0

Orijinal hedef dağılımı:
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

İkili hedef (0 = hasta değil, 1 = hasta):
num
0    164
1    139
Name: count, dtype: int64
Oranlar (%):
num
0    54.1
1    45.9
Name: proportion, dtype: float64

Sayısal değişkenlerin istatistikleri:
          count    mean    std    min    25%    50%    75%    max
age       303.0   54.44   9.04   29.0   48.0   56.0   61.0   77.0
trestbps  303.0  131.69  17.60   94.0  120.0  130.0  140.0  200.0
chol      303.0  246.69  51.78  126.0  211.0  241.

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Grafikler kaydedildi -> ciktilar

StatlogHeart VERİ SETİ ANALİZİ
Satır, sütun sayısı: (270, 13)

Veri tipleri:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
dtype: object

Eksik değer sayısı (sütun bazında):
Hiç eksik değer yok

Tekrar eden satır sayısı: 0

Orijinal hedef dağılımı:
heart-disease
1    150
2    120
Name: count, dtype: int64

İkili hedef (0 = hasta değil, 1 = hasta):
heart-disease
0    150
1    120
Name: count, dtype: int64
Oranlar (%):
heart-disease
0    55.6
1    44.4
Name: proportion, dtype: float64

Sayısal değişkenlerin istatistikleri:
          count    mean    std    min    25%    50%    75%    max
age       270.0   54.43   9.11   29.0   48.0   55.0   61.0   77.0
trestbps  270.0  131.34  17.86   94.0  120.0  130.0  140.0  200.0
chol      270.0  249.66  5

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Grafikler kaydedildi -> ciktilar


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24264\3441273949.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Veri Ön İşleme (Preprocessing) ve Veri Setinin Bölünmesi

In [25]:
def hazirla(X, y, ad):
    print("\n" + "=" * 60)
    print(ad, "ÖN İŞLEME")
    print("=" * 60)

    # %70 eğitim, %15 doğrulama, %15 test (stratify=y ile sınıf oranı korunuyor)
    X_egitim, X_gecici, y_egitim, y_gecici = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42)
    X_dogrulama, X_test, y_dogrulama, y_test = train_test_split(
        X_gecici, y_gecici, test_size=0.50, stratify=y_gecici, random_state=42)

    print("Eğitim:", X_egitim.shape[0], "| Doğrulama:", X_dogrulama.shape[0],
          "| Test:", X_test.shape[0])

    sayisal_adim = Pipeline([
        ("doldur", SimpleImputer(strategy="median")),
        ("olcekle", StandardScaler())
    ])
    kategorik_adim = Pipeline([
        ("doldur", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    donusturucu = ColumnTransformer([
        ("sayisal", sayisal_adim, SAYISAL),
        ("kategorik", kategorik_adim, KATEGORIK)
    ])

    # Veri sızıntısını önlemek için fit işlemi sadece eğitim verisinde yapılır
    donusturucu.fit(X_egitim)
    egitim_son = donusturucu.transform(X_egitim)
    dogrulama_son = donusturucu.transform(X_dogrulama)
    test_son = donusturucu.transform(X_test)

    sutun_adlari = donusturucu.get_feature_names_out()

    print("\n--- ÖN İŞLEME ÖNCESİ / SONRASI ---")
    print("Veri sayısı        :", len(X), "->", len(X))
    print("Özellik sayısı     :", X.shape[1], "->", egitim_son.shape[1])
    print("Eksik değer        :", int(X.isnull().sum().sum()), "->",
          int(np.isnan(egitim_son).sum()))
    print("Eğitim ortalaması (sayısal, ölçekleme sonrası):",
          np.round(egitim_son[:, :5].mean(axis=0), 3))
    print("Eğitim std (sayısal, ölçekleme sonrası)      :",
          np.round(egitim_son[:, :5].std(axis=0), 3))

    for isim, M, etiket in [("egitim", egitim_son, y_egitim),
                            ("dogrulama", dogrulama_son, y_dogrulama),
                            ("test", test_son, y_test)]:
        df = pd.DataFrame(M, columns=sutun_adlari)
        df["hedef"] = etiket.values
        df.to_csv(f"{KLASOR}/{ad}_{isim}_hazir.csv", index=False)

    print("\nİlk 3 eğitim satırı:")
    print(pd.DataFrame(egitim_son, columns=sutun_adlari).head(3).round(2))

hazirla(X1, y1, "HeartDisease")
hazirla(X2, y2, "StatlogHeart")


HeartDisease ÖN İŞLEME
Eğitim: 212 | Doğrulama: 45 | Test: 46

--- ÖN İŞLEME ÖNCESİ / SONRASI ---
Veri sayısı        : 303 -> 303
Özellik sayısı     : 13 -> 28
Eksik değer        : 6 -> 0
Eğitim ortalaması (sayısal, ölçekleme sonrası): [-0. -0. -0. -0.  0.]
Eğitim std (sayısal, ölçekleme sonrası)      : [1. 1. 1. 1. 1.]

İlk 3 eğitim satırı:
   sayisal__age  sayisal__trestbps  sayisal__chol  sayisal__thalach  \
0         -0.29               0.36          -0.98              0.88   
1          0.49               1.21          -0.69              0.33   
2         -2.20              -0.49          -1.05              1.11   

   sayisal__oldpeak  kategorik__sex_0.0  kategorik__sex_1.0  \
0             -0.80                 1.0                 0.0   
1              0.54                 0.0                 1.0   
2             -0.89                 0.0                 1.0   

   kategorik__cp_1.0  kategorik__cp_2.0  kategorik__cp_3.0  ...  \
0                0.0                0.0           